<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта: Система способов оплаты</h2>

----

### Вариант задания № 16


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Разработка системы различных способов оплаты на базе класса PaymentMethod. Система предусматривает онлайн-оплату, банковский перевод и оплату наличными.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;
// [НОВОЕ СПРИНТ 5] Делегат для вызова методов отображения деталей
public delegate void PaymentDetailsDelegate();

// Интерфейс для отправки чека
public interface IReceiptable
{
    void SendReceipt(string email);
}

// Интерфейс для возврата средств
public interface IRefundable
{
    void Refund(decimal amount);
}

// Базовый класс
public class PaymentMethod
{
    private int _paymentMethodId;
    private string _methodName;
    private decimal _minAmount;
    private decimal _commission;
    private decimal _maxAmount;
    private bool _isActive;
    private DateTime _lastTransactionDate;

    // [НОВОЕ СПРИНТ 5] Делегат и событие для уведомлений о статусе платежей
    public delegate void PaymentHandler(string message);
    public event PaymentHandler PaymentNotification;

    public int PaymentMethodId
    {
        get { return _paymentMethodId; }
        set { _paymentMethodId = value; }
    }

    public string MethodName
    {
        get { return _methodName; }
        set { _methodName = value; }
    }

    public decimal MinAmount
    {
        get { return _minAmount; }
        set { _minAmount = value; }
    }

    public decimal Commission
    {
        get { return _commission; }
        set { _commission = value; }
    }

    public string Currency { get; set; }

    public decimal MaxAmount
    {
        get { return _maxAmount; }
        set { _maxAmount = value; }
    }

    public bool IsActive
    {
        get { return _isActive; }
        set { _isActive = value; }
    }

    public DateTime LastTransactionDate
    {
        get { return _lastTransactionDate; }
        set { _lastTransactionDate = value; }
    }

    // [НОВОЕ СПРИНТ 5] 3 новых атрибута в базовом классе
    public string ProviderName { get; set; }
    public int TransactionCount { get; set; }
    public string SupportPhone { get; set; }

    public PaymentMethod(int paymentMethodId, string methodName,
        decimal minAmount, decimal commission, decimal maxAmount = 1000000m)
    {
        PaymentMethodId = paymentMethodId;
        MethodName = methodName;
        MinAmount = minAmount;
        Commission = commission;
        Currency = "RUB";
        MaxAmount = maxAmount;
        IsActive = true;
        ProviderName = "FinTech Провайдер";
        TransactionCount = 0;
        SupportPhone = "8-800-555-35-35";
    }

    // [НОВОЕ СПРИНТ 5] Переопределение Equals и GetHashCode для работы с HashSet
    public override bool Equals(object obj)
    {
        if (obj is PaymentMethod other)
        {
            return PaymentMethodId == other.PaymentMethodId && MethodName == other.MethodName;
        }
        return false;
    }

    public override int GetHashCode()
    {
        return HashCode.Combine(PaymentMethodId, MethodName);
    }

    public bool CheckMinimumAmount(decimal amount)
    {
        return amount >= MinAmount;
    }

    public bool CheckMaximumAmount(decimal amount)
    {
        return amount <= MaxAmount;
    }

    public decimal CalculateCommission(decimal amount)
    {
        return amount * (Commission / 100);
    }

    // [НОВОЕ СПРИНТ 5] Новые методы в базовом классе + вызов события
    public void UpdateCommission(decimal newCommission)
    {
        Commission = newCommission;
        PaymentNotification?.Invoke($"[Событие] Комиссия для '{MethodName}' изменена на {newCommission}%");
    }

    public void ToggleActiveStatus()
    {
        IsActive = !IsActive;
        PaymentNotification?.Invoke($"[Событие] Статус активности '{MethodName}' изменен на: {IsActive}");
    }

    public void ResetTransactionCounter()
    {
        TransactionCount = 0;
        Console.WriteLine($"Счетчик транзакций для '{MethodName}' сброшен.");
    }

    public void CancelPayment()
    {
        Console.WriteLine($"Платеж по способу '{MethodName}' отменен.");
        PaymentNotification?.Invoke($"[Событие] Отмена платежа через '{MethodName}'.");
    }

    public virtual bool ProcessPayment(decimal amount)
    {
        if (!IsActive)
        {
            Console.WriteLine($"Способ оплаты '{MethodName}' временно неактивен.");
            return false;
        }

        if (!CheckMaximumAmount(amount))
        {
            Console.WriteLine(
                $"Ошибка: сумма {amount} {Currency} превышает максимальную ({MaxAmount} {Currency}).");
            return false;
        }

        if (!CheckMinimumAmount(amount))
        {
            Console.WriteLine(
                $"Ошибка: сумма {amount} {Currency} меньше минимальной " +
                $"({MinAmount} {Currency}) для '{MethodName}'.");
            return false;
        }

        decimal commission = CalculateCommission(amount);
        LastTransactionDate = DateTime.Now;
        TransactionCount++;

        Console.WriteLine(
            $"Успешно: платеж на сумму {amount} {Currency} " +
            $"(комиссия: {commission} {Currency}) проведен через '{MethodName}'.");

        // Вызываем событие при успешном платеже
        PaymentNotification?.Invoke($"[Событие] Проведен платеж {amount} {Currency} через '{MethodName}'. Всего операций: {TransactionCount}");

        return true;
    }

    public bool ProcessPayment(decimal amount, string comment)
    {
        bool result = ProcessPayment(amount);
        Console.WriteLine($"Комментарий к платежу: {comment}");
        return result;
    }

    public virtual void GetPaymentDetails()
    {
        Console.WriteLine(
            $"ID: {PaymentMethodId} | Название: {MethodName} | " +
            $"Мин: {MinAmount} {Currency} | Макс: {MaxAmount} {Currency} | " +
            $"Комиссия: {Commission}% | Активен: {IsActive} | Операций: {TransactionCount}");
    }
}

// Онлайн-оплата
public class OnlinePayment : PaymentMethod, IReceiptable
{
    private string _paymentUrl;

    public string PaymentUrl
    {
        get { return _paymentUrl; }
        set { _paymentUrl = value; }
    }

    // [НОВОЕ СПРИНТ 5] Новые атрибуты в производном классе
    public bool IsSslSecured { get; set; }
    public int TimeoutSeconds { get; set; }

    public OnlinePayment(int paymentMethodId, string methodName,
        decimal minAmount, decimal commission, string paymentUrl)
        : base(paymentMethodId, methodName, minAmount, commission)
    {
        PaymentUrl = paymentUrl;
        IsSslSecured = true;
        TimeoutSeconds = 60;
    }

    public void SendReceipt(string email)
    {
        Console.WriteLine($"Электронный чек отправлен на почту: {email}");
    }

    // [НОВОЕ СПРИНТ 5] Новый метод в производном классе
    public void CheckConnection()
    {
        Console.WriteLine($"Проверка соединения с {PaymentUrl} (SSL: {IsSslSecured}, Таймаут: {TimeoutSeconds} сек.) — ОК.");
    }

    public override void GetPaymentDetails()
    {
        base.GetPaymentDetails();
        Console.WriteLine($"URL для оплаты: {PaymentUrl} | SSL: {IsSslSecured}");
    }
}

// Банковский перевод
public class BankTransfer : PaymentMethod
{
    private string _bankData;

    public string BankData
    {
        get { return _bankData; }
        set { _bankData = value; }
    }

    // [НОВОЕ СПРИНТ 5] Новые атрибуты в производном классе
    public string Bic { get; set; }
    public int ProcessingDays { get; set; }

    public BankTransfer(int paymentMethodId, string methodName,
        decimal minAmount, decimal commission, string bankData, string bic = "044525225")
        : base(paymentMethodId, methodName, minAmount, commission)
    {
        BankData = bankData;
        Bic = bic;
        ProcessingDays = 3;
    }

    // [НОВОЕ СПРИНТ 5] Новый метод
    public void PrintBankRequisites()
    {
        Console.WriteLine($"Реквизиты: {BankData}, БИК: {Bic}, Срок перевода: до {ProcessingDays} дн.");
    }

    public override bool ProcessPayment(decimal amount)
    {
        if (!base.ProcessPayment(amount))
            return false;

        Console.WriteLine($"Перевод отправлен по реквизитам: {BankData} (БИК: {Bic})");
        return true;
    }
}

// Оплата наличными
public class CashPayment : PaymentMethod
{
    private string _cashPickupPoint;

    public string CashPickupPoint
    {
        get { return _cashPickupPoint; }
        set { _cashPickupPoint = value; }
    }

    // [НОВОЕ СПРИНТ 5] Новые атрибуты
    public string CashierName { get; set; }
    public bool HasChangeAvailable { get; set; }

    public CashPayment(int paymentMethodId, string methodName,
        decimal minAmount, decimal commission, string cashPickupPoint, string cashierName = "Смирнова А.В.")
        : base(paymentMethodId, methodName, minAmount, commission)
    {
        CashPickupPoint = cashPickupPoint;
        CashierName = cashierName;
        HasChangeAvailable = true;
    }

    // [НОВОЕ СПРИНТ 5] Новый метод
    public void PrintCashierInfo()
    {
        Console.WriteLine($"Точка: {CashPickupPoint}, Кассир: {CashierName}, Размен в наличии: {HasChangeAvailable}");
    }

    public override bool ProcessPayment(decimal amount)
    {
        if (!base.ProcessPayment(amount))
            return false;

        Console.WriteLine($"Ожидайте оплату наличными в точке: {CashPickupPoint} (Кассир: {CashierName})");
        return true;
    }
}

// Оплата банковской картой
public class CardPayment : OnlinePayment, IRefundable
{
    public string CardNumber { get; set; }
    public string CardHolder { get; set; }

    // [НОВОЕ СПРИНТ 5] Новые атрибуты
    public string PaymentSystem { get; set; }
    public decimal CashbackPercent { get; set; }

    public CardPayment(int paymentMethodId, string methodName,
        decimal minAmount, decimal commission, string paymentUrl,
        string cardNumber, string cardHolder, string paymentSystem = "МИР")
        : base(paymentMethodId, methodName, minAmount, commission, paymentUrl)
    {
        CardNumber = cardNumber;
        CardHolder = cardHolder;
        PaymentSystem = paymentSystem;
        CashbackPercent = 1.5m;
    }

    // [НОВОЕ СПРИНТ 5] Новый метод расчета кэшбэка
    public decimal CalculateCashback(decimal amount)
    {
        decimal cashback = amount * (CashbackPercent / 100);
        Console.WriteLine($"Начислен кэшбэк ({CashbackPercent}%): {cashback} {Currency} по карте {PaymentSystem}");
        return cashback;
    }

    public void Refund(decimal amount)
    {
        Console.WriteLine(
            $"Оформлен возврат {amount} {Currency} на карту {CardNumber} ({CardHolder})");
    }

    public void Refund(decimal amount, string reason)
    {
        Refund(amount);
        Console.WriteLine($"Причина возврата: {reason}");
    }

    public override bool ProcessPayment(decimal amount)
    {
        if (!base.ProcessPayment(amount))
            return false;

        Console.WriteLine($"Списание с карты {CardNumber} ({PaymentSystem}), держатель: {CardHolder}");
        CalculateCashback(amount);
        return true;
    }
}

// Generic класс для хранения и обработки коллекции способов оплаты
public class PaymentCollection<T> where T : PaymentMethod
{
    private List<T> _payments = new List<T>();

    public void Add(T payment)
    {
        _payments.Add(payment);
    }

    public List<T> GetItems()
    {
        return _payments;
    }

    public void ProcessAll(decimal amount)
    {
        foreach (var payment in _payments)
        {
            payment.ProcessPayment(amount);
        }
    }

    public void ShowAllDetails()
    {
        foreach (var payment in _payments)
        {
            payment.GetPaymentDetails();
        }
    }
}

// Класс заказа
public class Order
{
    public int OrderId { get; set; }
    public decimal TotalAmount { get; set; }

    public Order(int orderId, decimal totalAmount)
    {
        OrderId = orderId;
        TotalAmount = totalAmount;
    }

    public void Pay(PaymentMethod paymentMethod)
    {
        Console.WriteLine(
            $"\nПопытка оплаты заказа №{OrderId} на сумму {TotalAmount} руб.");

        bool success = paymentMethod.ProcessPayment(TotalAmount);
        Console.WriteLine(success
            ? $"Заказ №{OrderId} успешно оплачен."
            : $"Заказ №{OrderId} НЕ оплачен.");
    }
}

// =========================================================================
// ТЕСТИРОВАНИЕ И ДЕМОНСТРАЦИЯ (Коллекции, Делегаты, Лямбда-выражения, События)
// =========================================================================

// --- Создание объектов ---
BankTransfer bank = new BankTransfer(
    1, "Банковский перевод", 500, 1.5m,
    "ПАО Сбербанк, Р/С 407...");

CashPayment cash = new CashPayment(
    2, "Наличные", 50, 0m,
    "Касса №1 (ул. Мира, 10)");

CardPayment card = new CardPayment(
    3, "Банковская карта онлайн", 100, 2.0m,
    "https://securepay.ru",
    "2200-****-****-1234",
    "Иван Иванов");

// --- 1. Работа с СОБЫТИЯМИ ---
Console.WriteLine("=== 1. Работа с событиями ===");
static void OnPaymentNotification(string message)
{
    Console.WriteLine(message);
}

// Подписываемся на события объектов
card.PaymentNotification += OnPaymentNotification;
bank.PaymentNotification += OnPaymentNotification;

card.UpdateCommission(1.8m); // Вызовет событие смены комиссии
card.ProcessPayment(1500);   // Вызовет событие успешной оплаты
bank.CancelPayment();        // Вызовет событие отмены платежа

// --- 2. Работа с ДЕЛЕГАТАМИ ---
Console.WriteLine("\n=== 2. Работа с делегатами ===");
PaymentDetailsDelegate showCardDetails = card.GetPaymentDetails;
PaymentDetailsDelegate showCashDetails = cash.GetPaymentDetails;

showCardDetails();
showCashDetails();

// Проверка новых методов производных классов
card.CheckConnection();
bank.PrintBankRequisites();
cash.PrintCashierInfo();

// --- 3. Работа с КОЛЛЕКЦИЯМИ (List, Dictionary, HashSet) и ЛЯМБДА-ВЫРАЖЕНИЯМИ ---
Console.WriteLine("\n=== 3. Работа со списком (List) и Лямбда-выражениями ===");
List<PaymentMethod> paymentList = new List<PaymentMethod> { bank, cash, card };

// Используем лямбда-выражение для поиска способов оплаты с комиссией меньше 1.6%
var lowCommissionMethods = paymentList.Where(p => p.Commission < 1.6m);

Console.WriteLine("Способы оплаты с комиссией менее 1.6%:");
foreach (var method in lowCommissionMethods)
{
    method.GetPaymentDetails();
}

Console.WriteLine("\n=== 4. Работа со словарем (Dictionary) ===");
Dictionary<string, PaymentMethod> paymentDict = new Dictionary<string, PaymentMethod>();
paymentDict.Add("BANK_SBER", bank);
paymentDict.Add("CASH_MAIN", cash);
paymentDict.Add("CARD_MIR", card);

if (paymentDict.TryGetValue("CARD_MIR", out PaymentMethod foundMethod))
{
    Console.WriteLine("Найден способ оплаты по ключу 'CARD_MIR':");
    foundMethod.GetPaymentDetails();
}

Console.WriteLine("\n=== 5. Работа с множеством уникальных элементов (HashSet) ===");
HashSet<PaymentMethod> paymentSet = new HashSet<PaymentMethod>();
paymentSet.Add(bank);
paymentSet.Add(cash);
paymentSet.Add(card);

// Попытка добавить дубликат (с тем же ID = 3 и названием)
CardPayment duplicateCard = new CardPayment(
    3, "Банковская карта онлайн", 100, 2.0m,
    "https://securepay.ru", "2200-****-****-1234", "Иван Иванов");

bool isDuplicateAdded = paymentSet.Add(duplicateCard);
Console.WriteLine($"Был ли добавлен дубликат карты в HashSet? {isDuplicateAdded}"); // false

// --- 6. Взаимодействие с классом Order ---
Console.WriteLine("\n=== 6. Оплата заказов (Order) ===");
Order order1 = new Order(101, 1000);
order1.Pay(card);

=== 1. Работа с событиями ===
[Событие] Комиссия для 'Банковская карта онлайн' изменена на 1.8%
Успешно: платеж на сумму 1500 RUB (комиссия: 27.000 RUB) проведен через 'Банковская карта онлайн'.
[Событие] Проведен платеж 1500 RUB через 'Банковская карта онлайн'. Всего операций: 1
Списание с карты 2200-****-****-1234 (МИР), держатель: Иван Иванов
Начислен кэшбэк (1.5%): 22.500 RUB по карте МИР
Платеж по способу 'Банковский перевод' отменен.
[Событие] Отмена платежа через 'Банковский перевод'.

=== 2. Работа с делегатами ===
ID: 3 | Название: Банковская карта онлайн | Мин: 100 RUB | Макс: 1000000 RUB | Комиссия: 1.8% | Активен: True | Операций: 1
URL для оплаты: https://securepay.ru | SSL: True
ID: 2 | Название: Наличные | Мин: 50 RUB | Макс: 1000000 RUB | Комиссия: 0% | Активен: True | Операций: 0
Проверка соединения с https://securepay.ru (SSL: True, Таймаут: 60 сек.) — ОК.
Реквизиты: ПАО Сбербанк, Р/С 407..., БИК: 044525225, Срок перевода: до 3 дн.
Точка: Касса №1 (ул. Мира, 10), Касс